In [2]:
import pandas as pd
import re
from transformers import AutoTokenizer
from datasets import Dataset

# ==========================================
# 1. LOAD DATASET SINTETIK
# ==========================================
file_name = "../../data/dataset_report_generator.csv"
try:
    df = pd.read_csv(file_name)
    print(f"✅ Dataset berhasil dimuat! Total baris: {df.shape[0]}")
except FileNotFoundError:
    print(f"❌ File {file_name} tidak ditemukan. Pastikan kamu sudah men-generate datanya.")

# ==========================================
# 2. PREPROCESSING KHUSUS NLG (Natural Language Generation)
# ==========================================
def clean_text_nlg(text):
    if pd.isna(text):
        return ""
    
    text = str(text)
    # Menghapus spasi ganda, enter (\n), dan tab (\t)
    text = re.sub(r'\s+', ' ', text)
    
    # Menghapus karakter aneh, TAPI tetap mempertahankan:
    # Huruf (w), Angka, Spasi (s), Titik, Koma, Tanda Seru/Tanya, Tanda Hubung, dan Simbol [ ] | (untuk tag input)
    text = re.sub(r'[^\w\s\.,!?\-Rp\[\]\|]', '', text)
    
    # CATATAN PENTING: 
    # Kita TIDAK menggunakan Sastrawi (Stemming) atau menghapus Stopwords di sini!
    # AI butuh tata bahasa utuh ("di", "ke", "dan") agar bisa menulis laporan dengan natural.
    
    return text.strip()

# Mengaplikasikan fungsi pembersihan ke Kolom Input (X) dan Output (Y)
print("🧹 Memulai pembersihan teks...")
df['clean_input'] = df['linearized_data_input'].apply(clean_text_nlg)
df['clean_output'] = df['narasi_laporan_output'].apply(clean_text_nlg)

# Ubah format Pandas DataFrame menjadi HuggingFace Dataset agar lebih ringan di-proses GPU
hf_dataset = Dataset.from_pandas(df[['clean_input', 'clean_output']])


# ==========================================
# 3. TOKENIZATION (Sub-word to Numbers)
# ==========================================
# Kita ganti menggunakan mT5 (Multilingual T5), model Gen AI standar industri yang sangat stabil
model_checkpoint = "google/mt5-small"
print(f"🤖 Memuat tokenizer dari {model_checkpoint}...")

# Gunakan use_fast=False dan legacy=False untuk kompatibilitas mT5 terbaru
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=False, legacy=False)

# Fungsi untuk memecah teks menjadi angka tensor (Input ID & Labels)
def tokenize_function(examples):
    # 1. Tokenize Input (Teks Tabel Linearized)
    model_inputs = tokenizer(
        examples["clean_input"], 
        max_length=128,      # Panjang maksimal kata input
        truncation=True,     
        padding="max_length" 
    )
    
    # 2. Tokenize Target (Teks Laporan Narasi)
    # Untuk model teks generatif, target (Y) diperlakukan sebagai 'labels'
    labels = tokenizer(
        text_target=examples["clean_output"], 
        max_length=128, 
        truncation=True, 
        padding="max_length"
    )
    
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("🔢 Memulai Tokenisasi...")
# Terapkan tokenisasi secara paralel
tokenized_datasets = hf_dataset.map(tokenize_function, batched=True)

# Mengecek hasil
print("\n🎉 Tokenisasi Selesai! Berikut adalah bentuk Tensor dari Baris 1:")
print("\nInput Asli:", tokenized_datasets[0]['clean_input'])
print("Input IDs (Angka untuk GPU):", tokenized_datasets[0]['input_ids'][:15], "...") 

print("\nOutput Asli:", tokenized_datasets[0]['clean_output'])
print("Labels IDs (Kunci Jawaban GPU):", tokenized_datasets[0]['labels'][:15], "...")

✅ Dataset berhasil dimuat! Total baris: 676
🧹 Memulai pembersihan teks...
🤖 Memuat tokenizer dari google/mt5-small...


🔢 Memulai Tokenisasi...


Map: 100%|██████████| 676/676 [00:00<00:00, 3156.28 examples/s]


🎉 Tokenisasi Selesai! Berikut adalah bentuk Tensor dari Baris 1:

Input Asli: [MESIN] Pompa Air Utama Gedung A [STATUS] Prediksi Rusak 10 Hari [SEVERITY] Sedang [ESTIMASI_BIAYA] Rp 7.500.000
Input IDs (Angka untuk GPU): [491, 511, 192797, 439, 79366, 262, 1889, 259, 84133, 1874, 28413, 298, 491, 216127, 399] ...

Output Asli: Divisi Pemeliharaan melaporkan proyeksi kerusakan pada Pompa Air Utama Gedung A dalam 10 hari, dengan tingkat keparahan Sedang. Diperlukan alokasi anggaran sebesar Rp 7.500.000 untuk mitigasi risiko dan perbaikan.
Labels IDs (Kunci Jawaban GPU): [179373, 266, 15250, 69745, 321, 416, 89296, 502, 99569, 4702, 68018, 72737, 1404, 79366, 262] ...


In [3]:
import os
import mlflow
import dagshub
from transformers import (
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from peft import get_peft_model, LoraConfig, TaskType

# ==========================================
# 1. SETUP DAGSHUB & MLFLOW
# ==========================================
print("🔗 Menghubungkan ke DagsHub MLflow...")
# Ganti dengan repo_owner dan repo_name milikmu jika berbeda
dagshub.init(repo_owner='NazeeraAlthea', repo_name='exigen-smart-maintenance', mlflow=True)
mlflow.set_experiment("Report_Generator_mT5_LoRA")

# ==========================================
# 2. SPLIT DATASET (Train & Validation)
# ==========================================
print("🗂️ Membagi data menjadi Train (85%) dan Validation (15%)...")
# tokenized_datasets adalah variabel dari cell sebelumnya
split_dataset = tokenized_datasets.train_test_split(test_size=0.15, seed=42)
train_dataset = split_dataset['train']
eval_dataset = split_dataset['test']

# ==========================================
# 3. LOAD MODEL & APPLY LoRA
# ==========================================
# Pastikan menggunakan model_checkpoint yang sama dari cell Tokenizer ("google/mt5-small")
print(f"🤖 Memuat Model asli {model_checkpoint}...")
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

print("⚡ Memasang arsitektur LoRA (Low-Rank Adaptation)...")
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    inference_mode=False,
    r=8,               # Ukuran rank (semakin kecil = semakin hemat RAM)
    lora_alpha=32,
    lora_dropout=0.1
)
model = get_peft_model(model, lora_config)

# Menampilkan statistik betapa ringannya model yang dilatih
model.print_trainable_parameters() 

# ==========================================
# 4. TRAINING ARGUMENTS & COLLATOR
# ==========================================
# DataCollator bertugas menyatukan baris-baris data ke dalam batch
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="../../models/report_cost_generator/mt5-report-generator-logs",
    eval_strategy="epoch",           # DIUBAH: Gunakan eval_strategy, bukan evaluation_strategy
    save_strategy="epoch",           # Simpan model sementara tiap epoch
    learning_rate=1e-3,              # Learning rate untuk LoRA sedikit lebih besar dari biasanya
    per_device_train_batch_size=4,   # Jika laptop kehabisan RAM/VRAM (OOM), turunkan ini jadi 2 atau 1
    per_device_eval_batch_size=4,
    num_train_epochs=5,              # Coba 5 iterasi dulu untuk memastikan jalan
    weight_decay=0.01,
    logging_steps=5,                 # Catat performa ke MLflow setiap 5 step
    report_to="mlflow",              # Ajaib! HuggingFace otomatis melempar grafik ke MLflow
    load_best_model_at_end=True,     # Otomatis memakai model terbaik di akhir
)

# ==========================================
# 5. INITIALIZE TRAINER & TRAIN
# ==========================================
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
)

print("\n🚀 MEMULAI PROSES FINE-TUNING...")
with mlflow.start_run() as run:
    trainer.train()
    
    # 6. SIMPAN MODEL TERBAIK
    save_path = "../../models/report_cost_generator/best_report_generator_lora"
    
    # Pastikan folder models/ ada
    if not os.path.exists("../../models/report_cost_generator"):
        os.makedirs("../../models/report_cost_generator")
        
    model.save_pretrained(save_path)
    tokenizer.save_pretrained(save_path)
    
    print(f"\n✅ Selesai! Model dan Tokenizer terbaik berhasil disimpan di: {save_path}")
    print(f"📊 Silakan buka Dashboard DagsHub kamu untuk melihat grafik evaluasinya.")

🔗 Menghubungkan ke DagsHub MLflow...


Initialized MLflow to track repo "NazeeraAlthea/exigen-smart-maintenance"

Repository NazeeraAlthea/exigen-smart-maintenance initialized!

🗂️ Membagi data menjadi Train (85%) dan Validation (15%)...
🤖 Memuat Model asli google/mt5-small...


Loading weights: 100%|██████████| 192/192 [00:00<00:00, 22241.12it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


⚡ Memasang arsitektur LoRA (Low-Rank Adaptation)...
trainable params: 344,064 || all params: 300,520,832 || trainable%: 0.1145

🚀 MEMULAI PROSES FINE-TUNING...


d:\05_Personal\College\semester-6\NTG-Project\exigen-smart-maintenance\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,8.663504,7.144626
2,7.135360,6.166039
3,4.537775,3.754395
4,3.946331,3.052528
5,4.104376,2.985561


d:\05_Personal\College\semester-6\NTG-Project\exigen-smart-maintenance\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
d:\05_Personal\College\semester-6\NTG-Project\exigen-smart-maintenance\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
d:\05_Personal\College\semester-6\NTG-Project\exigen-smart-maintenance\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
d:\05_Personal\College\semester-6\NTG-Project\exigen-smart-maintenance\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is s


✅ Selesai! Model dan Tokenizer terbaik berhasil disimpan di: ../../models/report_cost_generator/best_report_generator_lora
📊 Silakan buka Dashboard DagsHub kamu untuk melihat grafik evaluasinya.
🏃 View run caring-hog-42 at: https://dagshub.com/NazeeraAlthea/exigen-smart-maintenance.mlflow/#/experiments/2/runs/22695c9a694a403f8c3a8695d3a6a255
🧪 View experiment at: https://dagshub.com/NazeeraAlthea/exigen-smart-maintenance.mlflow/#/experiments/2


In [5]:
import torch
import re
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel

# ==========================================
# 1. LOAD MODEL TERBAIK (Base Model + LoRA)
# ==========================================
save_path = "../../models/report_cost_generator/best_report_generator_lora"
base_model_name = "google/mt5-small"

print("🤖 Memuat Tokenizer dan Model AI hasil Fine-Tuning...\n")
# Load Tokenizer yang sudah kita save
tokenizer = AutoTokenizer.from_pretrained(save_path)

# Trik LoRA: Load otak aslinya (mT5), lalu pasangkan "ingatan tambahan" (LoRA) milikmu
base_model = AutoModelForSeq2SeqLM.from_pretrained(base_model_name)
model = PeftModel.from_pretrained(base_model, save_path)

# ==========================================
# 2. FUNGSI GENERATOR LAPORAN
# ==========================================
def buat_laporan_keuangan(data_prediksi_tabel):
    # a. Pembersihan Teks (Sama seperti Tahap 2)
    teks_bersih = re.sub(r'\s+', ' ', str(data_prediksi_tabel))
    teks_bersih = re.sub(r'[^\w\s\.,!?\-Rp\[\]\|]', '', teks_bersih).strip()
    
    # b. Tokenisasi (Ubah ke angka)
    inputs = tokenizer(teks_bersih, return_tensors="pt", max_length=128, truncation=True)
    
    # c. Proses Generation (AI sedang berpikir)
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        max_new_tokens=150,     # Panjang maksimal laporan
        num_beams=4,            # Beam Search: Teknik agar grammar kalimatnya logis & tidak terputus
        temperature=0.2,        # Temperature 0.2: Sangat tegas, formal, dan faktual (karena untuk Akuntan)
        do_sample=True,
        early_stopping=True
    )
    
    # d. Decode (Ubah angka kembali menjadi bahasa manusia)
    laporan = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return laporan

# ==========================================
# 3. MARI KITA UJI COBA!
# ==========================================
# Simulasi: Ini adalah data yang ditangkap otomatis dari model Melvin, Jose, dan Najma
data_testing_1 = "[MESIN] Genset Pabrik [STATUS] Prediksi Rusak 7 Hari [SEVERITY] Kritis [ESTIMASI_BIAYA] Rp 25.000.000"
data_testing_2 = "[MESIN] AC Sentral Lantai 3 [STATUS] Prediksi Rusak 20 Hari [SEVERITY] Rendah [ESTIMASI_BIAYA] Rp 150.000"

print("📥 DATA INPUT 1 (Dari Pipeline):")
print(data_testing_1)
print("\n📝 HASIL LAPORAN EKSEKUTIF (BUATAN AI):")
print(buat_laporan_keuangan(data_testing_1))

print("\n--------------------------------------------------\n")

print("📥 DATA INPUT 2 (Dari Pipeline):")
print(data_testing_2)
print("\n📝 HASIL LAPORAN EKSEKUTIF (BUATAN AI):")
print(buat_laporan_keuangan(data_testing_2))

🤖 Memuat Tokenizer dan Model AI hasil Fine-Tuning...



Loading weights: 100%|██████████| 192/192 [00:00<00:00, 32074.97it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


📥 DATA INPUT 1 (Dari Pipeline):
[MESIN] Genset Pabrik [STATUS] Prediksi Rusak 7 Hari [SEVERITY] Kritis [ESTIMASI_BIAYA] Rp 25.000.000

📝 HASIL LAPORAN EKSEKUTIF (BUATAN AI):
ingkat <extra_id_51>an 

--------------------------------------------------

📥 DATA INPUT 2 (Dari Pipeline):
[MESIN] AC Sentral Lantai 3 [STATUS] Prediksi Rusak 20 Hari [SEVERITY] Rendah [ESTIMASI_BIAYA] Rp 150.000

📝 HASIL LAPORAN EKSEKUTIF (BUATAN AI):
inlash <extra_id_9>    
